# Naive Bayes Sentiment Analysis Model
# Introduction

This notebook implements a Naive Bayes model to perform sentiment classification on Amazon product reviews. The goal is to evaluate how effectively textual review data can predict sentiment and whether incorporating additional metadata features improves model performance.

Two experiments are conducted: one using only TF-IDF text features and another combining TF-IDF features with review metadata.

## Naive Bayes Model

Naive Bayes is a probabilistic machine learning algorithm based on Bayes' theorem. 
It assumes that input features are conditionally independent given the class label, 
which simplifies computation and makes it efficient for high-dimensional text data. (Mitchel,1997)

Because of its efficiency and strong performance on text classification tasks, 
Naive Bayes is commonly used for sentiment analysis and spam detection.

In [1]:
# Install Neccsary libraries
!pip install joblib pandas scipy scikit-learn

In [2]:
#import neccesary libraries

# save and load trained machine learning models.
import joblib

# handling datasets
import pandas as pd

# Provides scientific computing tools
from scipy.sparse import hstack

import os


# Machine learning library
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

## Load all prepared files

This section loads:
- text features
- metadata features
- training labels
- testing labels
- TF-IDF vectorizer

These files were prepared earlier in the preprocessing pipeline.

In [4]:
# Load TF-IDF features

X_train_text = joblib.load("shared/X_train_text.pkl")
X_test_text = joblib.load("shared/X_test_text.pkl")

# Load metadata features
X_train_meta = joblib.load("shared/X_train_meta.pkl")
X_test_meta = joblib.load("shared/X_test_meta.pkl")

# Load labels
y_train = joblib.load("shared/y_train.pkl")
y_test = joblib.load("shared/y_test.pkl")

## Inspect loaded data

This step checks the data types and shapes of the loaded objects to confirm that the inputs are ready for modelling.

In [7]:
print("Data types")
print("X_train_text:", type(X_train_text))
print("X_test_text :", type(X_test_text))
print("X_train_meta:", type(X_train_meta))
print("X_test_meta :", type(X_test_meta))
print("y_train     :", type(y_train))
print("y_test      :", type(y_test))

print("\nShapes and lengths")
print("X_train_text shape:", X_train_text.shape)
print("X_test_text shape :", X_test_text.shape)

if hasattr(X_train_meta, "shape"):
    print("X_train_meta shape:", X_train_meta.shape)

if hasattr(X_test_meta, "shape"):
    print("X_test_meta shape :", X_test_meta.shape)

print("y_train length:", len(y_train))
print("y_test length :", len(y_test))

Data types
X_train_text: <class 'scipy.sparse._csr.csr_matrix'>
X_test_text : <class 'scipy.sparse._csr.csr_matrix'>
X_train_meta: <class 'pandas.core.frame.DataFrame'>
X_test_meta : <class 'pandas.core.frame.DataFrame'>
y_train     : <class 'pandas.core.series.Series'>
y_test      : <class 'pandas.core.series.Series'>

Shapes and lengths
X_train_text shape: (661918, 10000)
X_test_text shape : (165480, 10000)
X_train_meta shape: (661918, 3)
X_test_meta shape : (165480, 3)
y_train length: 661918
y_test length : 165480


## Prepare metadata features

The TF-IDF text features are stored in sparse format.  
To combine metadata with text features, the metadata must also be converted into sparse format.

In [10]:
# Convert metadata to sparse matrix
if isinstance(X_train_meta, pd.DataFrame):
    X_train_meta_sparse = csr_matrix(X_train_meta.values)
else:
    X_train_meta_sparse = csr_matrix(X_train_meta)

if isinstance(X_test_meta, pd.DataFrame):
    X_test_meta_sparse = csr_matrix(X_test_meta.values)
else:
    X_test_meta_sparse = csr_matrix(X_test_meta)

print("Metadata converted to sparse matrices successfully.")
print("X_train_meta_sparse shape:", X_train_meta_sparse.shape)
print("X_test_meta_sparse shape :", X_test_meta_sparse.shape)

Metadata converted to sparse matrices successfully.
X_train_meta_sparse shape: (661918, 3)
X_test_meta_sparse shape : (165480, 3)


## Define evaluation function

This function calculates:
- Accuracy
- Precision
- Recall
- F1-score

It also prints a classification report for each experiment.

In [8]:
def evaluate_model(y_true, y_pred, experiment_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n{experiment_name}")
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        "experiment": experiment_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

## Experiment A: Text Only

This experiment trains a Naive Bayes model using only TF-IDF text features.

### Features used
- X_train_text
- X_test_text

In [10]:
# Train model using text features only
nb_text_only = MultinomialNB()
nb_text_only.fit(X_train_text, y_train)

# Predict on test set
y_pred_text_only = nb_text_only.predict(X_test_text)

# Evaluate model
results_text_only = evaluate_model(
    y_test,
    y_pred_text_only,
    "Experiment A: Text Only"
)


Experiment A: Text Only
--------------------------------------------------
Accuracy : 0.9102
Precision: 0.9037
Recall   : 0.9102
F1 Score : 0.9004

Classification Report:
              precision    recall  f1-score   support

    negative       0.81      0.49      0.61     23530
    positive       0.92      0.98      0.95    141950

    accuracy                           0.91    165480
   macro avg       0.86      0.73      0.78    165480
weighted avg       0.90      0.91      0.90    165480



## Experiment A: Text-Only Sentiment Classification

The Naive Bayes model was trained using TF-IDF text features extracted from the review content. The model achieved an overall accuracy of 0.91 and an F1 score of 0.90, indicating strong performance in classifying sentiment from textual information.

However, the classification report reveals a noticeable class imbalance in the dataset. While the model performs very well in identifying positive reviews (F1 = 0.95), it struggles more with negative reviews (F1 = 0.61), with a recall of 0.49. This suggests that a significant number of negative reviews are misclassified as positive, likely due to the higher proportion of positive reviews in the dataset.

These results indicate that while the model is effective overall, performance on the minority class (negative reviews) could potentially be improved by addressing the class imbalance in the data.

## Experiment B: Text + Metadata

This experiment combines text features and metadata features.

### Features used
- TF-IDF text features
- review_length
- vote
- verified

This experiment tests whether adding metadata improves classification performance.

In [12]:
# Combine text and metadata features
X_train_combined = hstack([X_train_text, X_train_meta])
X_test_combined = hstack([X_test_text, X_test_meta])

print("Combined feature matrices created successfully.")
print("X_train_combined shape:", X_train_combined.shape)
print("X_test_combined shape :", X_test_combined.shape)

Combined feature matrices created successfully.
X_train_combined shape: (661918, 10003)
X_test_combined shape : (165480, 10003)


In [14]:
# Train model using text + metadata
nb_text_meta = MultinomialNB()
nb_text_meta.fit(X_train_combined, y_train)

# Predict on test set
y_pred_text_meta = nb_text_meta.predict(X_test_combined)

# Evaluate model
results_text_meta = evaluate_model(
    y_test,
    y_pred_text_meta,
    "Experiment B: Text + Metadata"
)


Experiment B: Text + Metadata
--------------------------------------------------
Accuracy : 0.8843
Precision: 0.87
Recall   : 0.8843
F1 Score : 0.8633

Classification Report:
              precision    recall  f1-score   support

    negative       0.72      0.30      0.43     23530
    positive       0.89      0.98      0.94    141950

    accuracy                           0.88    165480
   macro avg       0.81      0.64      0.68    165480
weighted avg       0.87      0.88      0.86    165480



## Experiment B: Text + Metadata

In this experiment, metadata features (review length, word count, and short review flag) were combined with TF-IDF text features to train the Naive Bayes classifier. However, the results show that incorporating metadata reduced the overall performance of the model.

The accuracy decreased from 0.91 (text only) to 0.88 (text + metadata), and the F1-score also dropped from 0.90 to 0.86. In particular, the model’s ability to correctly identify negative reviews deteriorated significantly, with recall decreasing from 0.49 to 0.30.

These results suggest that Naive Bayes performs better when trained purely on textual features, likely because the algorithm assumes feature independence and is primarily designed for word frequency based representations such as TF-IDF.

## Compare experiment results

This section combines the results from both experiments into a single table for direct comparison.

In [15]:
results_df = pd.DataFrame([results_text_only, results_text_meta])

print("Comparison of experiment results:")
results_df

Comparison of experiment results:


,experiment,accuracy,precision,recall,f1_score
0,Experiment A: Text Only,0.910158,0.903691,0.910158,0.900449
1,Experiment B: Text + Metadata,0.884258,0.869952,0.884258,0.863334


## Identify the best-performing experiment

This section identifies which experiment achieved the highest F1-score.

In [18]:
best_experiment = results_df.loc[results_df["f1_score"].idxmax()]

print("Best Performing Experiment\n")
print("Experiment:", best_experiment["experiment"])
print("Accuracy  :", round(best_experiment["accuracy"], 4))
print("Precision :", round(best_experiment["precision"], 4))
print("Recall    :", round(best_experiment["recall"], 4))
print("F1 Score  :", round(best_experiment["f1_score"], 4))

Best Performing Experiment

Experiment: Experiment A: Text Only
Accuracy  : 0.9102
Precision : 0.9037
Recall    : 0.9102
F1 Score  : 0.9004


## Save results to CSV

The results from both experiments are saved to:

`results/naive_bayes_results.csv`

In [21]:
# create folder if it doesn't exist
os.makedirs("models", exist_ok=True)

# store experiment results
results = pd.DataFrame({
    "Model": ["Naive Bayes", "Naive Bayes"],
    "Features": ["TF-IDF", "TF-IDF + Metadata"],
    "Accuracy": [0.9102, 0.8843],
    "F1 Score": [0.9004, 0.8633]
})

# save results
results.to_csv("models/naive_bayes_results.csv", index=False)

# save trained models
joblib.dump(nb_text_only, "models/naive_bayes_tfidf.pkl")
joblib.dump(nb_text_meta, "models/naive_bayes_combined.pkl")

print("Naive Bayes results and models saved successfully.")

Naive Bayes results and models saved successfully.


## References

The implementation used the following Python tools and libraries:

- joblib for loading saved feature files and labels
- pandas for storing and exporting the results table
- scikit-learn for training and evaluating the Naive Bayes model
- scipy.sparse for combining sparse text and metadata features

Main functions and classes used:
- MultinomialNB
- accuracy_score
- precision_score
- recall_score
- f1_score
- classification_report
- hstack
- csr_matrix